The RAG process is fixed: every request is used as the basis for a database search for matching documents. The information found in the matching documents is then used to provide a detailed output.
With agents, the LLM agent is given the *option* to perform a database search for answering a given prompt.

In [16]:
from dotenv import load_dotenv
from openai import OpenAI
from ingest import load_faq_data, build_index
import json


In [6]:
load_dotenv()
openai_client = OpenAI()

In [7]:
documents = load_faq_data()
index = build_index(documents)

In [8]:
def search(query):
	boost_dict = {"question": 3.0, "section": 0.5}
	filter_dict = {"course": "llm-zoomcamp"}

	return index.search(
		query,
		num_results=5,
		boost_dict=boost_dict,
		filter_dict=filter_dict
  )

In [23]:
def build_context(search_results):
	lines = []

	for doc in search_results:
		lines.append(doc['section'])
		lines.append('Q: ' + doc['question'])
		lines.append('A: ' + doc['answer'])
		lines.append('')

	return '\n'.join(lines).strip()

In [24]:
PROMPT_TEMPLATE = """
	QUESTION: {question}

	CONTEXT:
	{context}
""".strip()

In [25]:
def build_prompt(query, search_results):
	context = build_context(search_results)
	return PROMPT_TEMPLATE.format(
		question=query, context=context
	)

In [12]:
search_tool = {
	"type": "function",
	'function': {
		"name": "search",
		"description": "Search the FAQ database for entries matching the given query.",
		"parameters": {
			"type": "object",
			"properties": {
				"query": {
					"type": "string",
					"description": "Search query text to look up in the course FAQ."
				}
			},
			"required": ["query"],
			"additionalProperties": False
		}
	}
}

In [13]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [14]:
response = openai_client.chat.completions.create(
	model="gpt-4o-mini",
	messages=messages,
	tools=[search_tool],
	user="llm-zoomcamp",
	stream=False
)

In [32]:
if response.choices[0].finish_reason == "tool_calls":
	function_call = response.choices[0].message.tool_calls[0].function

	if function_call.name == "search":
		args = json.loads(function_call.arguments)
		search_results = search(*args)


BadRequestError: Error code: 400 - {'error': {'message': "OpenAI API error: 400 Missing required parameter: 'messages[1].role'.", 'type': 'api_error'}}

In [31]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'type': 'function_call_output',
  'output': '[\n  {\n    "id": "83da208a64",\n    "course": "llm-zoomcamp",\n    "section": "Module 1 Homework",\n    "question": "Homework: Returning Empty list after filtering my query (HW Q3)",\n    "answer": "This is likely to be an error when indexing the data. First, you need to add the index settings before adding the data to the indices, then you will be good to go applying your filters and query."\n  },\n  {\n    "id": "95feb4e75b",\n    "course": "llm-zoomcamp",\n    "section": "Module 2: Vector Search",\n    "question": "Why does cosine similarity reduce to a matrix multiplication between the embeddings and the query vector?",\n    "answer": "Cosine similarity measures how aligned two vectors are, regardless of their magnitude. When all vectors (including the query) are normalized to unit length, their magnitudes no longer matter. In this case, cosine similarity is